## Mixed Integer Linear Programming

Consider the following example, 

Given system constraints:

                                          2x + 4y >= 230 
                                          3x + 2y <= 190 
                                          x >= 0, x is integer
                                          y >= 0, y is continuious

Maximize objective function:
                            
                                          f(x) = 5x + 3y 


You need to find numbers for x and y in such a way that it satisfies constraints and maximizes the objective function.

### Environment Setup

First, let's check if we have a GPU available in system. This is only applicable if you are running server through this notebook. Else comment the next cell.

In [ ]:
import subprocess
import html
from IPython.display import display, HTML

def check_gpu():
    try:
        result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=5)
        result.check_returncode()
        lines = result.stdout.splitlines()
        gpu_info = lines[2] if len(lines) > 2 else "GPU detected"
        gpu_info_escaped = html.escape(gpu_info)
        display(HTML(f"""
        <div style="border:2px solid #4CAF50;padding:10px;border-radius:10px;background:#e8f5e9;">
            <h3>✅ GPU is enabled</h3>
            <pre>{gpu_info_escaped}</pre>
        </div>
        """))
        return True
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired, FileNotFoundError, IndexError) as e:
        display(HTML("""
        <div style="border:2px solid red;padding:15px;border-radius:10px;background:#ffeeee;">
            <h3>⚠️ GPU not detected!</h3>
            <p>This notebook requires a <b>GPU runtime</b>.</p>
            
            <h4>If running in Google Colab:</h4>
            <ol>
              <li>Click on <b>Runtime → Change runtime type</b></li>
              <li>Set <b>Hardware accelerator</b> to <b>GPU</b></li>
              <li>Then click <b>Save</b> and <b>Runtime → Restart runtime</b>.</li>
            </ol>
            
            <h4>If running in Docker:</h4>
            <ol>
              <li>Ensure you have <b>NVIDIA Docker runtime</b> installed (<code>nvidia-docker2</code>)</li>
              <li>Run container with GPU support: <code>docker run --gpus all ...</code></li>
              <li>Or use: <code>docker run --runtime=nvidia ...</code> for older Docker versions</li>
              <li>Verify GPU access: <code>docker run --gpus all nvidia/cuda:12.0.0-base-ubuntu22.04 nvidia-smi</code></li>
            </ol>
            
            <p><b>Additional resources:</b></p>
            <ul>
              <li><a href="https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/install-guide.html" target="_blank">NVIDIA Container Toolkit Installation Guide</a></li>
            </ul>
        </div>
        """))
        return False

check_gpu()

#### Install dependencies



In [ ]:
# Enable this in case you are running this in google colab or such places where cuOpt is not yet installed
#!pip uninstall -y cuda-python cuda-bindings cuda-core
#!pip install --upgrade --extra-index-url=https://pypi.nvidia.com cuopt-server-cu12 nvidia-nvjitlink-cu12
#!pip install --upgrade --extra-index-url=https://pypi.nvidia.com cuopt-server-cu13 nvidia-nvjitlink-cu13


#### Run cuOpt gRPC server in background

Please disable the next cell in case you are trying to test against a server running on a different machine.


In [ ]:
import subprocess
import atexit
import psutil
import time
from cuopt.grpc.linear_programming import Client, GrpcError

host = "localhost"
port = 5001

def is_cuopt_grpc_server_running():
    for proc in psutil.process_iter(["name", "cmdline"]):
        try:
            cmdline = proc.info["cmdline"] or []
            if any("cuopt_grpc_server" in cmd for cmd in cmdline):
                return True
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    return False

def wait_for_server_health(timeout=60):
    """Wait until cuopt_grpc_server answers a short ping."""
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            Client(host, port, tls=False).ping(timeout_seconds=2)
            print("Server is healthy!")
            return True
        except (GrpcError, OSError):
            pass

        print(
            "Waiting for server to be ready... "
            f"({int(time.time() - start_time)}s elapsed)"
        )
        time.sleep(1)

    print(f"Server health check timed out after {timeout} seconds")
    return False

# Only start the server if it is not already running
if not is_cuopt_grpc_server_running():
    server_process = subprocess.Popen(
        ["cuopt_grpc_server", "--port", str(port), "--workers", "1"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    def cleanup_server():
        try:
            server_process.terminate()
            server_process.wait(timeout=5)
            print("Terminated cuOpt gRPC server")
        except (subprocess.TimeoutExpired, ProcessLookupError):
            try:
                server_process.kill()
                print("Killed cuOpt gRPC server")
            except ProcessLookupError:
                pass

    atexit.register(cleanup_server)
    print("Started cuOpt gRPC server")
else:
    print("cuOpt gRPC server is already running")

if not wait_for_server_health():
    print("Warning: Server may not be fully ready")


### Build the model

Build the model with `Problem`, the algebraic API. `Client.submit` also accepts a `DataModel`; that form is in `mixed-integer-linear-programming-with-datamodel.ipynb`.


In [ ]:
from cuopt.linear_programming.problem import (
    CONTINUOUS,
    INTEGER,
    MAXIMIZE,
    Problem,
)
from cuopt.linear_programming.solver_settings import SolverSettings

problem = Problem("mixed_integer_linear_programming")
settings = SolverSettings()


### Add variables

`x` is an integer and `y` is continuous. Both are non-negative.


In [ ]:
x = problem.addVariable(lb=0, vtype=INTEGER, name="x")
y = problem.addVariable(lb=0, vtype=CONTINUOUS, name="y")


### Add constraints

```
2x + 4y >= 230
3x + 2y <= 190
```


In [ ]:
problem.addConstraint(2 * x + 4 * y >= 230, name="min_requirement")
problem.addConstraint(3 * x + 2 * y <= 190, name="capacity")


### Set the objective

Maximize `5x + 3y`.


In [ ]:
problem.setObjective(5 * x + 3 * y, sense=MAXIMIZE)


### Set Solver Configuration

The solver configuration can be fine-tuned for optimization and runtimes.


In [ ]:
settings.set_parameter("time_limit", 1.0)


### Solve the Problem

Submit the `Problem` to `cuopt_grpc_server` with the gRPC linear programming client.

The client returns values for `x` and `y`.

The following example shows how to use a locally hosted server:


In [ ]:
from cuopt.grpc.linear_programming import Client, JobStatus

# If cuOpt is not running on localhost:5001, edit host and port.
# wait() blocks until the job finishes or the call fails.
client = Client(host, port, tls=False)

job_id = client.submit(problem, settings)
try:
    status = client.wait(job_id)
    if status != JobStatus.COMPLETED:
        raise RuntimeError(f"job did not complete: {status}")
    solution = client.result(job_id, variable_names=["x", "y"])
finally:
    client.delete(job_id)

print("Termination:", solution.get_termination_reason())
print("Objective:  ", solution.get_primal_objective())
print("Variables:  ", solution.get_vars())


``Optimal`` termination means an optimal solution is available.


```text
Termination: Optimal
Objective:   303.5
Variables:   {'x': 37.0, 'y': 39.5}
```


### License

SPDX-FileCopyrightText: Copyright (c) 2025 NVIDIA CORPORATION & AFFILIATES. All rights reserved.

SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.